In [6]:
import os
import random
import shutil
import hashlib
from PIL import Image

MIN_SIZE = 64
N_POR_FORMATO = 40

# Origen absoluto; entrenamiento y destino relativos al proyecto.
RUTAS = {
    "jpg": (
        r"D:\TrabajoGrado\Imagenes\jpg_select",
        "Imagenes/jpg",
        "Imagenes_prueba/jpg",
        (".jpg", ".jpeg"),
    ),
    "png": (
        r"D:\TrabajoGrado\Imagenes\png_select",
        "Imagenes/png",
        "Imagenes_prueba/png",
        (".png",),
    ),
    "webp": (
        r"D:\TrabajoGrado\Imagenes\webp_select",
        "Imagenes/webp",
        "Imagenes_prueba/webp",
        (".webp",),
    ),
}


def hash_archivo(ruta):
    sha = hashlib.sha256()

    with open(ruta, "rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            sha.update(bloque)

    return sha.hexdigest()


def nombre_normalizado(nombre):
    return os.path.splitext(nombre)[0].strip().casefold()


def imagen_valida(ruta):
    try:
        with Image.open(ruta) as imagen:
            imagen.verify()

        with Image.open(ruta) as imagen:
            ancho, alto = imagen.size

        return ancho >= MIN_SIZE and alto >= MIN_SIZE

    except Exception:
        return False


def copiar_imagenes_prueba(n=N_POR_FORMATO, seed=42):
    rng = random.Random(seed)

    for formato, (origen, usadas, destino, extensiones) in RUTAS.items():

        # Verificar origen y conjunto utilizado para entrenamiento.
        for carpeta in (origen, usadas):
            if not os.path.isdir(carpeta):
                raise FileNotFoundError(
                    f"No existe la carpeta: {carpeta}"
                )

        os.makedirs(destino, exist_ok=True)

        # --------------------------------------------------
        # 1. Indexar las imágenes utilizadas para el modelo
        # --------------------------------------------------
        archivos_usados = [
            f for f in os.listdir(usadas)
            if os.path.isfile(os.path.join(usadas, f))
            and f.lower().endswith(extensiones)
        ]

        nombres_usados = {
            nombre_normalizado(f) for f in archivos_usados
        }

        hashes_usados = {
            hash_archivo(os.path.join(usadas, f))
            for f in archivos_usados
        }

        # --------------------------------------------------
        # 2. Comprobar las imágenes que ya existen en prueba
        # --------------------------------------------------
        existentes = [
            f for f in os.listdir(destino)
            if os.path.isfile(os.path.join(destino, f))
            and f.lower().endswith(extensiones)
        ]

        nombres_prueba = set()
        hashes_prueba = set()

        for archivo in existentes:
            ruta = os.path.join(destino, archivo)

            if not imagen_valida(ruta):
                raise ValueError(f"Imagen inválida en prueba: {ruta}")

            nombre = nombre_normalizado(archivo)
            sha = hash_archivo(ruta)

            if nombre in nombres_usados or sha in hashes_usados:
                raise ValueError(
                    f"Esta imagen de prueba coincide con una imagen "
                    f"utilizada para entrenamiento:\n{ruta}"
                )

            nombres_prueba.add(nombre)
            hashes_prueba.add(sha)

        if len(existentes) > n:
            raise ValueError(
                f"{destino} ya contiene {len(existentes)} imágenes "
                f"{formato.upper()}; el objetivo es {n}."
            )

        faltantes = n - len(existentes)

        if faltantes == 0:
            print(f"[OK] {formato.upper()}: ya hay {n} imágenes.")
            continue

        # --------------------------------------------------
        # 3. Filtrar el origen contra entrenamiento y prueba
        # --------------------------------------------------
        archivos_origen = [
            f for f in os.listdir(origen)
            if os.path.isfile(os.path.join(origen, f))
            and f.lower().endswith(extensiones)
        ]

        candidatas = []
        excluidas_nombre = 0
        excluidas_hash = 0
        descartadas = 0

        for archivo in archivos_origen:
            ruta = os.path.join(origen, archivo)
            nombre = nombre_normalizado(archivo)

            # Mismo nombre en entrenamiento o prueba.
            if nombre in nombres_usados or nombre in nombres_prueba:
                excluidas_nombre += 1
                continue

            if not imagen_valida(ruta):
                descartadas += 1
                continue

            sha = hash_archivo(ruta)

            # Contenido idéntico al de una imagen ya existente.
            if sha in hashes_usados or sha in hashes_prueba:
                excluidas_hash += 1
                continue

            candidatas.append(archivo)

        print("\n" + "=" * 65)
        print(f"FORMATO: {formato.upper()}")
        print(f"Origen: {origen}")
        print(f"Entrenamiento: {usadas}")
        print(f"Destino: {destino}")
        print(f"Imágenes existentes en prueba: {len(existentes)}")
        print(f"Excluidas por nombre: {excluidas_nombre}")
        print(f"Excluidas por contenido idéntico: {excluidas_hash}")
        print(f"Inválidas o demasiado pequeñas: {descartadas}")
        print(f"Candidatas disponibles: {len(candidatas)}")
        print(f"Imágenes por completar: {faltantes}")

        if len(candidatas) < faltantes:
            raise ValueError(
                f"No hay suficientes imágenes {formato.upper()}: "
                f"{len(candidatas)} disponibles y {faltantes} necesarias."
            )

        # --------------------------------------------------
        # 4. Seleccionar y copiar sin recodificar
        # --------------------------------------------------
        seleccion = rng.sample(candidatas, faltantes)

        for archivo in seleccion:
            ruta_origen = os.path.join(origen, archivo)
            ruta_destino = os.path.join(destino, archivo)

            if os.path.exists(ruta_destino):
                raise FileExistsError(
                    f"El archivo ya existe: {ruta_destino}"
                )

            shutil.copy2(ruta_origen, ruta_destino)

            nombres_prueba.add(nombre_normalizado(archivo))
            hashes_prueba.add(hash_archivo(ruta_destino))

        print(f"[OK] Se copiaron {faltantes} imágenes.")
        print(f"[OK] Total de {formato.upper()} en prueba: {n}")

    print("\nSelección terminada: hasta 40 imágenes por formato.")


copiar_imagenes_prueba(n=40, seed=42)


FORMATO: JPG
Origen: D:\TrabajoGrado\Imagenes\jpg_select
Entrenamiento: Imagenes/jpg
Destino: Imagenes_prueba/jpg
Imágenes existentes en prueba: 0
Excluidas por nombre: 10000
Excluidas por contenido idéntico: 0
Inválidas o demasiado pequeñas: 0
Candidatas disponibles: 195
Imágenes por completar: 40
[OK] Se copiaron 40 imágenes.
[OK] Total de JPG en prueba: 40

FORMATO: PNG
Origen: D:\TrabajoGrado\Imagenes\png_select
Entrenamiento: Imagenes/png
Destino: Imagenes_prueba/png
Imágenes existentes en prueba: 0
Excluidas por nombre: 10000
Excluidas por contenido idéntico: 166
Inválidas o demasiado pequeñas: 2
Candidatas disponibles: 3272
Imágenes por completar: 40
[OK] Se copiaron 40 imágenes.
[OK] Total de PNG en prueba: 40

FORMATO: WEBP
Origen: D:\TrabajoGrado\Imagenes\webp_select
Entrenamiento: Imagenes/webp
Destino: Imagenes_prueba/webp
Imágenes existentes en prueba: 0
Excluidas por nombre: 10000
Excluidas por contenido idéntico: 0
Inválidas o demasiado pequeñas: 0
Candidatas disponible